In [60]:
import pandas as pd
import numpy as np
import matplotlib as plt

data_main = pd.read_csv('sources/upload-weeklyshipcrossingsforsixmaritimepassagesofinterest.csv')
data_excel = pd.read_excel(
    'sources/uktradeflowsofcontainerisedproductsthroughglobalmaritimepassages20202024.xlsx',
    sheet_name=['1.Monthly Volumes All (Imports)', '2.Monthly Volumes All (Exports)']
)
data_second = pd.concat(data_excel.values(), ignore_index=True)

In [61]:
data_main.head()

,Passage,Week of entry,Number of crossings
0,Bab-Al Mandab Strait,03/01/2022,394
1,Bab-Al Mandab Strait,10/01/2022,391
2,Bab-Al Mandab Strait,17/01/2022,386
3,Bab-Al Mandab Strait,24/01/2022,394
4,Bab-Al Mandab Strait,31/01/2022,415


In [62]:
data_main.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 695 entries, 0 to 694
Data columns (total 3 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Passage              695 non-null    object
 1   Week of entry        695 non-null    object
 2   Number of crossings  695 non-null    int64 
dtypes: int64(1), object(2)
memory usage: 16.4+ KB


In [63]:
data_second.head()

,Passage,Direction,TEU,Year,Month
0,Taiwan Strait,import,13625.1,2020,1
1,Taiwan Strait,import,2710.4,2020,2
2,Taiwan Strait,import,11166.5,2020,3
3,Taiwan Strait,import,7766,2020,4
4,Taiwan Strait,import,11118.4,2020,5


In [64]:
data_second.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Passage    600 non-null    object
 1   Direction  600 non-null    object
 2   TEU        600 non-null    object
 3   Year       600 non-null    int64 
 4   Month      600 non-null    int64 
dtypes: int64(2), object(3)
memory usage: 23.6+ KB


In [65]:
# 1. Convert "Week of entry" to datetime (day/month/year format)
data_main['Week of entry'] = pd.to_datetime(data_main['Week of entry'], format='%d/%m/%Y')

# 2. Group by Passage and Month, summing the weekly crossings
data_main = data_main.groupby(
    [data_main['Passage'], data_main['Week of entry'].dt.to_period('M')]
).agg({'Number of crossings': 'sum'}).reset_index()

# Rename for clarity
data_main.rename(columns={'Week of entry': 'Month'}, inplace=True)

data_main.head(10)


,Passage,Month,Number of crossings
0,Bab-Al Mandab Strait,2022-01,1980
1,Bab-Al Mandab Strait,2022-02,1625
2,Bab-Al Mandab Strait,2022-03,1661
3,Bab-Al Mandab Strait,2022-04,1710
4,Bab-Al Mandab Strait,2022-05,2100
5,Bab-Al Mandab Strait,2022-06,1785
6,Bab-Al Mandab Strait,2022-07,1836
7,Bab-Al Mandab Strait,2022-08,2319
8,Bab-Al Mandab Strait,2022-09,1824
9,Bab-Al Mandab Strait,2022-10,2335


In [66]:
data_main.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 167 entries, 0 to 166
Data columns (total 3 columns):
 #   Column               Non-Null Count  Dtype    
---  ------               --------------  -----    
 0   Passage              167 non-null    object   
 1   Month                167 non-null    period[M]
 2   Number of crossings  167 non-null    int64    
dtypes: int64(1), object(1), period[M](1)
memory usage: 4.0+ KB


In [67]:
# 1. Replace "x" in TEU with NaN, then convert to numeric
data_second['TEU'] = pd.to_numeric(data_second['TEU'].replace('x', np.nan))

# 2. Combine Year and Month into a datetime column
data_second['Date'] = pd.to_datetime(data_second[['Year', 'Month']].assign(day=1))

# 3. Group by Passage and Month, summing TEU
data_second = data_second.groupby(
    ['Passage', data_second['Date'].dt.to_period('M')]
).agg({'TEU': 'sum'}).reset_index()

data_second.head(10)

C:\Users\rakad\AppData\Local\Temp\ipykernel_9808\3138249727.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data_second['TEU'] = pd.to_numeric(data_second['TEU'].replace('x', np.nan))


,Passage,Date,TEU
0,Cape of Good Hope,2020-01,511.2
1,Cape of Good Hope,2020-02,334.7
2,Cape of Good Hope,2020-03,730.3
3,Cape of Good Hope,2020-04,1493.4
4,Cape of Good Hope,2020-05,4806.4
5,Cape of Good Hope,2020-06,2669.0
6,Cape of Good Hope,2020-07,5794.9
7,Cape of Good Hope,2020-08,4294.9
8,Cape of Good Hope,2020-09,1857.6
9,Cape of Good Hope,2020-10,1045.2


In [68]:
data_second = data_second[data_second['Date'] >= '2022-01']
data_second.head(10)

,Passage,Date,TEU
24,Cape of Good Hope,2022-01,776.4
25,Cape of Good Hope,2022-02,706.0
26,Cape of Good Hope,2022-03,608.0
27,Cape of Good Hope,2022-04,936.8
28,Cape of Good Hope,2022-05,991.4
29,Cape of Good Hope,2022-06,2712.9
30,Cape of Good Hope,2022-07,2080.5
31,Cape of Good Hope,2022-08,2223.5
32,Cape of Good Hope,2022-09,1948.2
33,Cape of Good Hope,2022-10,1165.4


In [69]:
# Left join: keep all rows from data_main, add UK_TEU where available
data_merged = pd.merge(data_main, data_second, left_on=['Passage', 'Month'], right_on=['Passage', 'Date'], how='left')

# Drop the duplicate date column and rename TEU
data_merged = data_merged.drop(columns=['Date']).rename(columns={'TEU': 'UK_TEU'})

data_merged.head(10)

,Passage,Month,Number of crossings,UK_TEU
0,Bab-Al Mandab Strait,2022-01,1980,NaN
1,Bab-Al Mandab Strait,2022-02,1625,NaN
2,Bab-Al Mandab Strait,2022-03,1661,NaN
3,Bab-Al Mandab Strait,2022-04,1710,NaN
4,Bab-Al Mandab Strait,2022-05,2100,NaN
5,Bab-Al Mandab Strait,2022-06,1785,NaN
6,Bab-Al Mandab Strait,2022-07,1836,NaN
7,Bab-Al Mandab Strait,2022-08,2319,NaN
8,Bab-Al Mandab Strait,2022-09,1824,NaN
9,Bab-Al Mandab Strait,2022-10,2335,NaN


In [70]:
data_merged.tail()

,Passage,Month,Number of crossings,UK_TEU
162,Taiwan Strait,2023-12,4515,8733.0
163,Taiwan Strait,2024-01,5693,9898.5
164,Taiwan Strait,2024-02,4259,11799.0
165,Taiwan Strait,2024-03,5233,11732.8
166,Taiwan Strait,2024-04,1178,21932.8


In [71]:
data_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 167 entries, 0 to 166
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype    
---  ------               --------------  -----    
 0   Passage              167 non-null    object   
 1   Month                167 non-null    period[M]
 2   Number of crossings  167 non-null    int64    
 3   UK_TEU               139 non-null    float64  
dtypes: float64(1), int64(1), object(1), period[M](1)
memory usage: 5.3+ KB


In [72]:
data_merged['UK_TEU'] = data_merged['UK_TEU'].replace('', np.nan)

In [73]:
data_merged.to_csv('sources/data_merged.csv', index=False)